# Approach 2 · Paired Disease Patch Attention
## 1. Setup

Use Python 3.13.17, CUDA 12.8 and the team repository's `requirements.txt`. Use `DEBUG=True` for a short train-only check. The experiment uses frozen DINOv2 features and trains lightweight heads.

Raw 333-way unseen Top-1 is the main endpoint. Previous F2 and R2 runs showed that most blocked unseen predictions lost to a different disease on the same crop. This experiment tests whether explicitly learning shared disease patch evidence changes that ranking.

The training idea draws on [Learning Attention as Disentangler, CVPR 2023](https://arxiv.org/abs/2303.15111).

In [ ]:
DEBUG = False

from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
import platform
import hashlib
import json
import sys

if sys.version_info[:3] != (3, 13, 17):
    raise RuntimeError(f"Select the Python 3.13.17 notebook kernel. Current kernel: {sys.executable} ({sys.version.split()[0]}).")

import pandas as pd
import timm
import torch
from IPython.display import display
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
BATCH_SIZE = 8 if DEBUG else 16
EPOCHS = 1 if DEBUG else 20
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 4
RUN_NAME = "approach2_paired_disease_attention" + ("_debug" if DEBUG else "")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = OUTPUT_DIR / "development"
FINAL_DIR = OUTPUT_DIR / "final"
OFFICIAL_DIR = OUTPUT_DIR / "official"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID

HEAD_LR, WEIGHT_DECAY = 1e-4, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.1, 0.1
GAMMAS = [step / 2 for step in range(25)]
FOLD_SEEDS = (43, 53, 67)
AUGMENTED_VIEWS = 1
PATCH_GRID, PATCH_WIDTH = 8, 128
PAIR_WEIGHT, ALIGNMENT_WEIGHT = 0.3, 0.1
RECIPES = {
    "A2_Control": {"patch_attention": False, "paired_supervision": False},
    "P0_SingleImage": {"patch_attention": True, "paired_supervision": False},
    "P1_PairedDisease": {"patch_attention": True, "paired_supervision": True},
}

requirements_path = ROOT / "requirements.txt"
requirements_bytes = requirements_path.read_bytes()
requirements = requirements_bytes.decode("utf-8-sig")
for entry in requirements.splitlines():
    entry = entry.split(" #", 1)[0].strip()
    if not entry or entry.startswith(("#", "--index-url", "--extra-index-url")):
        continue
    requirement = Requirement(entry)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    try:
        installed_version = metadata.version(requirement.name)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError(f"Install the repository requirements in this kernel: {entry}.") from error
    if not requirement.specifier.contains(installed_version, prereleases=True):
        raise RuntimeError(f"requirements.txt specifies {entry}, but this kernel has {installed_version}.")

if not DEBUG and (DEVICE.type != "cuda" or torch.version.cuda != "12.8"
                  or not torch.cuda.is_available()):
    raise RuntimeError("The full run requires the project CUDA 12.8 PyTorch build and a visible GPU.")

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "executable": sys.executable,
    "requirements_sha256": hashlib.sha256(requirements_bytes).hexdigest(),
    "cuda_build": torch.version.cuda,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "packages": dict(sorted((canonicalize_name(dist.metadata["Name"]), dist.version)
                            for dist in metadata.distributions() if dist.metadata["Name"])),
}

missing = [path for path, found in ((team.TRAIN_CSV, team.TRAIN_CSV.is_file()),
                                     (team.IMAGE_DIR, team.IMAGE_DIR.is_dir())) if not found]
if missing:
    raise FileNotFoundError(f"Missing dataset paths: {missing}. Copy the original dataset into the project data folder or run download_data.py.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the team's 323/333 common files.")
print(f"{DEVICE} | FP32 | batch {BATCH_SIZE} | {EPOCHS} epochs | DEBUG={DEBUG}")

## 2. Data

Use three predetermined train-only folds with the team's within-pair training split. Each fold withholds ten complete pairs from fit. Five guide tune and five provide a separate check within that fold. The folds can reuse some rows across folds.

The frozen backbone produces one clean and one augmented view per training image, with 8×8 pooled spatial tokens at 224 pixels. Cache each view once in CPU memory and store spatial tokens at FP16 precision (about 1.5 GiB total for 8,000 images). Every trial uses the same cache and batch order. For paired supervision, the donor has the same disease label and a different crop. Donor selection uses the current fold's fit rows only. Images with no eligible donor still receive the ordinary training losses.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")


train_transform, _ = get_transforms("imagenet")
train_hash = sha256_file(team.TRAIN_CSV)
row_by_image = {name: row for row, name in enumerate(train_full.image)}
pair_row = torch.as_tensor(train_full.pair_row.to_numpy(copy=True), dtype=torch.long)
if not torch.equal(train_pairs[pair_row], torch.tensor(train_full[["crop", "disease"]].to_numpy())):
    raise ValueError("Training image labels disagree with training pair rows.")


def frame_indices(frame):
    return torch.tensor([row_by_image[image_key(name)] for name in frame.image], dtype=torch.long)


def make_outer_fold(fit_frame, val_frame, seed):
    rows = fit_frame.groupby("pair_row", sort=True).agg(
        n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops, diseases = rows.crop.value_counts().to_dict(), rows.disease.value_counts().to_dict()
    held = []
    for row, item in rows[rows.n.ge(5)].sample(frac=1, random_state=seed).iterrows():
        if crops[item.crop] > 1 and diseases[item.disease] > 1:
            held.append(int(row))
            crops[item.crop] -= 1
            diseases[item.disease] -= 1
        if len(held) == 10:
            break
    if len(held) != 10:
        raise ValueError(f"Fold {seed} needs ten eligible training pairs.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=seed + 1)
    cut = len(seen) // 2
    groups = {
        "fit": frame_indices(fit_frame.loc[~fit_frame.pair_row.isin(held)]),
        "tune_seen": frame_indices(seen.iloc[:cut]),
        "check_seen": frame_indices(seen.iloc[cut:]),
        "tune_proxy": torch.where(torch.isin(pair_row, torch.tensor(held[:5])))[0],
        "check_proxy": torch.where(torch.isin(pair_row, torch.tensor(held[5:])))[0],
    }
    all_rows = torch.cat(list(groups.values()))
    if any(not len(rows) for rows in groups.values()) or len(all_rows) != len(all_rows.unique()):
        raise ValueError(f"Empty or overlapping groups in fold {seed}.")
    fit_mask = torch.bincount(pair_row[groups["fit"]], minlength=len(train_pairs)).bool()
    if torch.where(~fit_mask)[0].tolist() != sorted(held):
        raise ValueError(f"Fold {seed} has unexpected fit-pair coverage.")
    return {"seed": seed, "groups": groups, "fit_mask": fit_mask, "tune_pairs": held[:5], "check_pairs": held[5:]}


settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW", "schedule": "CosineAnnealingLR", "label_smoothing": SMOOTHING,
    "decorrelation_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE,
    "fold_seeds": list(FOLD_SEEDS), "augmented_views": AUGMENTED_VIEWS,
    "patch_grid": PATCH_GRID, "patch_width": PATCH_WIDTH,
    "pair_weight": PAIR_WEIGHT, "alignment_weight": ALIGNMENT_WEIGHT,
    "recipes": RECIPES, "calibration_grid": GAMMAS, "train_csv_sha256": train_hash,
    "requirements_sha256": environment["requirements_sha256"],
    "environment_sha256": hashlib.sha256(json.dumps(environment, sort_keys=True).encode()).hexdigest(),
    "environment_spec": {"python": "3.13.17", "requirements": "requirements.txt"}, "debug": DEBUG,
}

## 3. Model

| Arm | Design |
| --- | --- |
| A2 Control | Frozen DINOv2, crop and disease linear heads, logit decorrelation and Cross-crop SupCon |
| P0 Single Image | A2 plus a disease residual from learned attention over one image's patches |
| P1 Paired Disease | Same architecture as P0, with extra disease supervision on patches shared with a same-disease, different-crop training image |

P1 matches projected patch tokens by cosine similarity. A patch receives more weight when it has a close match in the donor. Both shared descriptions predict the training disease label. A stop-gradient alignment loss teaches the single-image pooling head to approximate the shared description. Inference uses only the input image.

The raw score sums crop and disease log-probabilities for each candidate. It uses the same function for every candidate.

In [ ]:
def new_backbone(pretrained):
    return timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0, img_size=team.IMAGE_SIZE)


def spatial_features(backbone, images, use_patches=True):
    if not use_patches:
        return backbone(images), None
    tokens = backbone.forward_features(images)
    global_features = backbone.forward_head(tokens)
    patches = tokens[:, backbone.num_prefix_tokens:]
    height, width = backbone.patch_embed.grid_size
    patches = patches.transpose(1, 2).reshape(len(images), -1, height, width)
    patches = F.adaptive_avg_pool2d(patches, (min(PATCH_GRID, height), min(PATCH_GRID, width)))
    return global_features, patches.flatten(2).transpose(1, 2)


class DiseaseNet(nn.Module):
    def __init__(self, spec, backbone):
        super().__init__()
        self.backbone = backbone.requires_grad_(False)
        width = backbone.num_features
        self.crop_head = nn.Linear(width, NUM_CROPS)
        self.disease_head = nn.Linear(width, NUM_DISEASES)
        self.use_patches = spec["patch_attention"]
        if self.use_patches:
            self.patch_encoder = nn.Sequential(nn.Linear(width, PATCH_WIDTH), nn.LayerNorm(PATCH_WIDTH))
            self.patch_attention = nn.Linear(PATCH_WIDTH, 1)
            self.patch_classifier = nn.Linear(PATCH_WIDTH, NUM_DISEASES, bias=False)

    def from_features(self, global_features, patches):
        crop, disease = self.crop_head(global_features), self.disease_head(global_features)
        if not self.use_patches:
            return crop, disease, None, None
        patch_features = self.patch_encoder(patches.float())
        weights = self.patch_attention(patch_features).softmax(dim=1)
        pooled = (weights * patch_features).sum(dim=1)
        return crop, disease + self.patch_classifier(pooled), patch_features, pooled

    def forward(self, images):
        global_features, patches = spatial_features(self.backbone, images, self.use_patches)
        if patches is not None:
            patches = patches.to(torch.float16)
        return self.from_features(global_features, patches)[:2]


def build_network(spec, pretrained, backbone=None):
    return DiseaseNet(spec, new_backbone(pretrained) if backbone is None else backbone)


def score_images(model, images, pairs):
    logits = model(images)
    return pair_scores(*logits, pairs), logits


def paired_disease_loss(model, query_patches, donor_patches, student, disease, criterion):
    donor = model.patch_encoder(donor_patches.float())
    similarity = F.normalize(query_patches, dim=-1) @ F.normalize(donor, dim=-1).transpose(1, 2)
    query_attention = (similarity.amax(dim=2) / TEMPERATURE).softmax(dim=1)
    donor_attention = (similarity.amax(dim=1) / TEMPERATURE).softmax(dim=1)
    query_shared = (query_attention[..., None] * query_patches).sum(dim=1)
    donor_shared = (donor_attention[..., None] * donor).sum(dim=1)
    loss = 0.5 * (criterion(model.patch_classifier(query_shared), disease)
                  + criterion(model.patch_classifier(donor_shared), disease))
    align = (1 - F.cosine_similarity(student, query_shared.detach(), dim=1)).mean()
    return PAIR_WEIGHT * loss + ALIGNMENT_WEIGHT * align


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()

## 4. Training

Select the recipe and epoch by average raw harmonic Top-1 on the three development tune folds. Break ties by proxy Top-1, seen Top-1, then earlier epoch. `raw_check_deltas.csv` compares P0 with A2 and P1 with P0 at P1's tune-selected epoch. Report the fold-local checks and donor coverage.

Gamma is tuned separately on training-derived folds for the supplementary calibrated report. It does not enter the training loss or raw model selection.

In [ ]:
class FeatureCache:
    def __init__(self, global_features, patches, rows):
        self.global_features = global_features
        self.patches = patches
        self.pos = {row: pos for pos, row in enumerate(rows.tolist())}

    def take(self, rows, use_patches):
        pos = [self.pos[row] for row in rows.tolist()]
        return self.global_features[pos].to(DEVICE), self.patches[pos].to(DEVICE) if use_patches else None


@torch.no_grad()
def cache_features(backbone, frame, transform, seed, name):
    team.set_seed(seed)
    loader = DataLoader(PlantDataset(frame, transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=torch.Generator().manual_seed(seed))
    global_cache = patch_cache = None
    offset = 0
    for step, (images, _, _, _) in enumerate(loader, 1):
        global_features, patches = spatial_features(backbone, images.to(DEVICE))
        if not torch.isfinite(global_features).all() or not torch.isfinite(patches).all():
            raise FloatingPointError(f"Invalid features in {name}.")
        if global_cache is None:
            global_cache = torch.empty((len(frame), global_features.shape[1]))
            patch_cache = torch.empty((len(frame), *patches.shape[1:]), dtype=torch.float16)
        end = offset + len(images)
        global_cache[offset:end] = global_features.cpu()
        patch_cache[offset:end] = patches.cpu()
        offset = end
        if step == 1 or step % 100 == 0 or step == len(loader):
            print(f"Feature cache {name}: {step}/{len(loader)}", flush=True)
    if offset != len(frame) or global_cache is None:
        raise RuntimeError(f"Cache {name} is incomplete.")
    return FeatureCache(global_cache, patch_cache, frame_indices(frame))


def donor_pool(indices):
    frame = train_full.iloc[indices.tolist()]
    pools = {}
    for disease, group in frame.groupby("disease"):
        if group.crop.nunique() < 2:
            continue
        for crop in group.crop.unique():
            pools[int(crop), int(disease)] = torch.tensor(group.index[group.crop.ne(crop)].tolist())
    return pools


def train_epoch(model, spec, optimizer, criterion, clean, views, indices, donors, epoch):
    generator = torch.Generator().manual_seed(team.SEED + epoch)
    donor_rng = torch.Generator().manual_seed(team.SEED + 1000 + epoch)
    cache = views[(epoch - 1) % len(views)]
    shuffled = indices[torch.randperm(len(indices), generator=generator)]
    loss_sum, paired = 0., 0
    for batch in shuffled.split(BATCH_SIZE):
        targets = train_pairs[pair_row[batch]]
        crop, disease = targets.to(DEVICE).unbind(1)
        crop_logits, disease_logits, regions, student = model.from_features(*cache.take(batch, model.use_patches))
        contrast, _ = cross_crop_supcon(center_logits(disease_logits), crop, disease)
        loss = (criterion(crop_logits, crop) + criterion(disease_logits, disease)
                + DECOR_WEIGHT * decorrelation(center_logits(crop_logits), center_logits(disease_logits))
                + SUPCON_WEIGHT * contrast)
        if spec["paired_supervision"]:
            links = [(j, donors[tuple(key)]) for j, key in enumerate(targets.tolist()) if tuple(key) in donors]
            if links:
                anchors = [j for j, _ in links]
                donor_rows = torch.stack([pool[torch.randint(len(pool), (), generator=donor_rng)] for _, pool in links])
                _, donor_patches = clean.take(donor_rows, True)
                loss = loss + paired_disease_loss(model, regions[anchors], donor_patches,
                                                   student[anchors], disease[anchors], criterion)
                paired += len(anchors)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        loss_sum += float(loss.detach()) * len(batch)
    return {"images_trained": len(indices), "train_loss": loss_sum / len(indices),
            "paired_images": paired, "paired_fraction": paired / len(indices)}


def make_optimizer(model):
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    return optimizer, scheduler


@torch.no_grad()
def score_group(model, clean, indices):
    pairs = train_pairs.to(DEVICE)
    pieces = []
    for batch in indices.split(BATCH_SIZE):
        logits = model.from_features(*clean.take(batch, model.use_patches))[:2]
        pieces.append(pair_scores(*logits, pairs).cpu())
    return torch.cat(pieces), pair_row[indices]


def split_scores(model, clean, fold, split):
    return tuple(score_group(model, clean, fold["groups"][f"{split}_{name}"])
                 for name in ("seen", "proxy"))


def rank_metrics(scores, fit_mask, gamma):
    (seen_score, seen_y), (proxy_score, proxy_y) = scores
    seen_score, proxy_score = (value - gamma * fit_mask for value in (seen_score, proxy_score))
    seen_top1 = top_k_accuracy(seen_score, seen_y, 1)
    proxy_top1 = top_k_accuracy(proxy_score, proxy_y, 1)
    proxy_rows = torch.where(~fit_mask)[0]
    best_proxy = proxy_rows[proxy_score[:, proxy_rows].argmax(1)]
    blocked = best_proxy.eq(proxy_y) & fit_mask[proxy_score.argmax(1)]
    margins = proxy_score.gather(1, proxy_y[:, None]).squeeze(1) - proxy_score[:, fit_mask].amax(1)
    return {"seen_top1": seen_top1, "proxy_top1": proxy_top1,
            "seen_top5": top_k_accuracy(seen_score, seen_y, 5),
            "proxy_top5": top_k_accuracy(proxy_score, proxy_y, 5),
            "harmonic_top1": harmonic_mean(seen_top1, proxy_top1),
            "blocked_by_seen_pct": round(float(blocked.float().mean()) * 100, 2),
            "median_true_pair_minus_top_seen": float(margins.median())}


def make_trial(backbone, spec):
    team.set_seed()
    return build_network(spec, pretrained=False, backbone=backbone).to(DEVICE).eval()


def head_snapshot(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()
            if not name.startswith("backbone.")}


def fit_trial(recipe, backbone, fold, clean, views):
    spec = RECIPES[recipe]
    model = make_trial(backbone, spec)
    indices = fold["groups"]["fit"]
    donors = donor_pool(indices) if spec["paired_supervision"] else {}
    if spec["paired_supervision"] and not donors:
        raise ValueError("No same-disease, different-crop fit donors.")
    optimizer, scheduler = make_optimizer(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    history, states, tune_scores = [], [], []
    for epoch in range(1, EPOCHS + 1):
        log = train_epoch(model, spec, optimizer, criterion, clean, views, indices, donors, epoch)
        scheduler.step()
        row = {"recipe": recipe, "fold": fold["seed"], "epoch": epoch,
               "donor_concept_groups": len(donors), **log}
        if not DEBUG:
            scores = split_scores(model, clean, fold, "tune")
            row.update(rank_metrics(scores, fold["fit_mask"], 0.0))
            tune_scores.append(scores)
            states.append(head_snapshot(model))
        history.append(row)
        print(row, flush=True)
    return history, states, tune_scores


def run_development():
    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    atomic_json(DEVELOPMENT_DIR / "environment.json", environment)
    (DEVELOPMENT_DIR / "requirements.txt").write_bytes(requirements_bytes)
    backbone = new_backbone(pretrained=True).to(DEVICE).requires_grad_(False).eval()
    train, val, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Expected train-only project loaders.")
    if DEBUG:
        pool = donor_pool(torch.arange(len(train_full)))
        if not pool:
            raise ValueError("No eligible cross-crop disease donor for Debug.")
        (crop, disease), donors = next(iter(pool.items()))
        anchors = train_full.index[train_full.crop.eq(crop) & train_full.disease.eq(disease)].tolist()
        selected = torch.tensor(sorted(set(anchors[:3] + donors[:3].tolist())), dtype=torch.long)
        frame = train_full.iloc[selected.tolist()]
        clean = cache_features(backbone, frame, image_transform, team.SEED, "debug clean")
        view = cache_features(backbone, frame, train_transform, team.SEED + 10, "debug augmented")
        fold = {"seed": FOLD_SEEDS[0], "groups": {"fit": selected}}
        logs = [row for recipe in RECIPES for row in fit_trial(recipe, backbone, fold, clean, [view])[0]]
        pd.DataFrame(logs).to_csv(DEVELOPMENT_DIR / "training_history.csv", index=False)
        return None, None, None, None

    folds = [make_outer_fold(train.dataset.frame, val.dataset.frame, seed) for seed in FOLD_SEEDS]
    atomic_json(DEVELOPMENT_DIR / "folds.json", [
        {"seed": fold["seed"], "tune_pairs": fold["tune_pairs"], "check_pairs": fold["check_pairs"],
         "image_counts": {k: len(v) for k, v in fold["groups"].items()}} for fold in folds])
    clean = cache_features(backbone, train_full, image_transform, team.SEED, "clean")
    views = [cache_features(backbone, train_full, train_transform, team.SEED + k + 10, f"train view {k + 1}")
             for k in range(AUGMENTED_VIEWS)]
    trials = {}
    history = []
    for recipe in RECIPES:
        for fold in folds:
            rows, states, scores = fit_trial(recipe, backbone, fold, clean, views)
            trials[recipe, fold["seed"]] = states, scores
            history.extend(rows)
    history = pd.DataFrame(history)
    history.to_csv(DEVELOPMENT_DIR / "training_history.csv", index=False)
    metrics = ["seen_top1", "proxy_top1", "seen_top5", "proxy_top5", "harmonic_top1",
               "blocked_by_seen_pct", "median_true_pair_minus_top_seen"]
    average = history.groupby(["recipe", "epoch"], as_index=False)[metrics].mean()
    average.to_csv(DEVELOPMENT_DIR / "mean_tune_metrics.csv", index=False)
    average = average.sort_values(["harmonic_top1", "proxy_top1", "seen_top1", "epoch"],
                                  ascending=[False, False, False, True], kind="stable")
    best = average.drop_duplicates("recipe")
    selection = {row["recipe"]: row for row in best.to_dict("records")}
    winner = best.iloc[0].recipe
    grids, checks = [], []
    gammas = {}
    matched_rows = []
    matched_index = selection["P1_PairedDisease"]["epoch"] - 1
    for recipe, choice in selection.items():
        index = choice["epoch"] - 1
        model = make_trial(backbone, RECIPES[recipe])
        recipe_grid = []
        for gamma in GAMMAS:
            per_fold = [rank_metrics(trials[recipe, fold["seed"]][1][index], fold["fit_mask"], gamma)
                        for fold in folds]
            means = pd.DataFrame(per_fold).mean().to_dict()
            recipe_grid.append({"recipe": recipe, "epoch": choice["epoch"], "gamma": gamma, **means})
        grids.extend(recipe_grid)
        gamma = max(recipe_grid, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                                  row["seen_top1"], -row["gamma"]))["gamma"]
        gammas[recipe] = gamma
        for fold in folds:
            state = trials[recipe, fold["seed"]][0][index]
            model.load_state_dict(state, strict=False)
            check = split_scores(model, clean, fold, "check")
            if matched_index == index:
                matched_check = check
            else:
                model.load_state_dict(trials[recipe, fold["seed"]][0][matched_index], strict=False)
                matched_check = split_scores(model, clean, fold, "check")
            matched_rows.append({"recipe": recipe, "fold": fold["seed"],
                                 **rank_metrics(matched_check, fold["fit_mask"], 0.0)})
            for name, penalty in (("raw", 0.0), ("calibrated", gamma)):
                checks.append({"recipe": recipe, "fold": fold["seed"], "epoch": choice["epoch"],
                               "scoring": name, "gamma": penalty, **rank_metrics(check, fold["fit_mask"], penalty)})
    pd.DataFrame(grids).to_csv(DEVELOPMENT_DIR / "calibration_tune_grid.csv", index=False)
    check = pd.DataFrame(checks)
    check.to_csv(DEVELOPMENT_DIR / "independent_checks.csv", index=False)
    raw = pd.DataFrame(matched_rows).set_index(["recipe", "fold"])[metrics]
    deltas = []
    for trial, control in (("P0_SingleImage", "A2_Control"), ("P1_PairedDisease", "P0_SingleImage")):
        for fold, row in (raw.loc[trial] - raw.loc[control]).iterrows():
            deltas.append({"recipe": trial, "reference": control, "fold": fold, "epoch": matched_index + 1,
                           **{f"{name}_delta": value for name, value in row.items()}})
    pd.DataFrame(deltas).to_csv(DEVELOPMENT_DIR / "raw_check_deltas.csv", index=False)
    selection_table = best.assign(gamma=best.recipe.map(gammas), selected=best.recipe.eq(winner))
    selection_table.to_csv(DEVELOPMENT_DIR / "selected_models.csv", index=False)
    display(selection_table)
    plan = {"recipe_selected_on_tune": winner, "settings": settings,
            "recipes": {key: {"selection": row, "gamma": gammas[key]} for key, row in selection.items()}}
    atomic_json(DEVELOPMENT_DIR / "selection_plan.json", plan)
    return plan, backbone, clean, views


plan, backbone, clean, views = run_development()

## 5. Final evaluation preparation

Train only the selected recipe on all training images with fresh heads and the original frozen backbone. Save its complete checkpoint and lock the selected epoch, gamma and checkpoint hash before opening official metadata.

In [ ]:
def read_final_lock(recipe):
    path = FINAL_DIR / "selection_lock.json"
    checkpoint = CHECKPOINT_DIR / "model.pt"
    lock = json.loads(path.read_text(encoding="utf-8"))
    if (lock["settings"] != settings or lock["recipe"] != recipe
            or lock["selection_plan"] != plan or lock["seen_pairs"] != train_pairs.tolist()
            or lock["checkpoint_sha256"] != sha256_file(checkpoint)):
        raise RuntimeError(f"Final checkpoint or plan changed for {recipe}.")
    return lock


def refit_recipe(recipe, spec, selection):
    model = make_trial(backbone, spec)
    indices = torch.arange(len(train_full))
    donors = donor_pool(indices) if spec["paired_supervision"] else {}
    if spec["paired_supervision"] and not donors:
        raise ValueError("No eligible donor in the full training set.")
    optimizer, scheduler = make_optimizer(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    epochs = selection["selection"]["epoch"]
    if not 1 <= epochs <= EPOCHS:
        raise ValueError(f"Invalid selected epoch for {recipe}.")
    history = []
    for epoch in range(1, epochs + 1):
        row = {"epoch": epoch, **train_epoch(model, spec, optimizer, criterion, clean, views,
                                            indices, donors, epoch)}
        scheduler.step()
        history.append(row)
        print(f"Full refit {recipe}: {row}", flush=True)
    checkpoint = CHECKPOINT_DIR / "model.pt"
    folder = FINAL_DIR
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    folder.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(folder / "refit_history.csv", index=False)
    refit = {"initialization": "original_pretrained_dinov2_fresh_heads",
             "data": "all_training_csv_rows", "images_per_epoch": len(train_full),
             "epochs": epochs, "cosine_t_max": EPOCHS}
    metadata = {"settings": settings, "recipe": recipe, "paired_supervision": spec["paired_supervision"],
                "recipe_spec": spec, "seen_pairs": train_pairs.tolist(), "selection": selection["selection"],
                "gamma": selection["gamma"], "refit": refit}
    atomic_save(checkpoint, {"state_dict": model.cpu().state_dict(), **metadata})
    lock = {**metadata, "selection_plan": plan, "checkpoint_sha256": sha256_file(checkpoint),
            "checkpoint_bytes": checkpoint.stat().st_size}
    atomic_json(folder / "selection_lock.json", lock)
    print(f"Locked checkpoint: {checkpoint.resolve()} ({checkpoint.stat().st_size / 1e6:.1f} MB)", flush=True)
    return lock


def refit_selected():
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development.")
    if json.loads((DEVELOPMENT_DIR / "selection_plan.json").read_text(encoding="utf-8")) != plan:
        raise RuntimeError("The development plan changed after selection.")
    recipe = plan["recipe_selected_on_tune"]
    selection = plan["recipes"][recipe]
    print(f"Full refit: {recipe} for {selection['selection']['epoch']} epochs", flush=True)
    if (FINAL_DIR / "selection_lock.json").is_file():
        return read_final_lock(recipe)
    return refit_recipe(recipe, RECIPES[recipe], selection)


winner_run = None if DEBUG else refit_selected()

## 6. Official test

Append ten unseen candidate identities to the 323 training identities. Generate the predictions for both scoring rules before using per-image labels. The primary row uses raw scores and `gamma=0`. Report seen, unseen and overall Top-1 and Top-5, harmonic mean, per-pair errors, and the existing robustness probes. The calibrated row is a supplementary comparison on the same checkpoint.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    mean, std = (torch.tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["imagenet"])
    parts, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        scores, (crop_logits, disease_logits) = score_images(model, images.to(DEVICE), pairs)
        scores = scores.cpu()
        if perturb is None:
            concepts.append(torch.stack((crop_logits.argmax(1), disease_logits.argmax(1)), dim=1).cpu())
        if scores.shape != (len(images), len(pairs)) or not torch.isfinite(scores).all():
            raise ValueError("Unexpected official candidate predictions.")
        parts.append(scores)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(parts), torch.cat(concepts) if concepts else None



def predict_recipe(recipe, run, pairs, filenames):
    saved = torch.load(CHECKPOINT_DIR / "model.pt", map_location="cpu", weights_only=True)
    for key in ("settings", "recipe", "recipe_spec", "paired_supervision", "selection", "gamma", "seen_pairs", "refit"):
        if saved[key] != run[key]:
            raise RuntimeError(f"Checkpoint and selection lock differ for {recipe}: {key}.")
    model = build_network(saved["recipe_spec"], pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model = model.to(DEVICE).eval()
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    for rule, gamma in (("raw", 0.0), ("calibrated", run["gamma"])):
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    return predictions, clean, heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, clean, (seen_heads, unseen_heads) = forward_pass
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    # Rows are unseen images and columns are candidate pairs.
    true_crop = pairs[y_unseen, 0]
    same_crop_candidates = true_crop[:, None].eq(pairs[None, :, 0])
    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    for name, prediction in predictions.items():
        gamma = prediction["gamma"]
        raw_scores = clean[1]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        final_margins = raw_margins + gamma
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"scoring": name, "gamma": gamma, "paired_supervision": run["paired_supervision"],
                   "selected_on_tune": run["recipe"] == plan["recipe_selected_on_tune"],
                   "epoch": run["selection"]["epoch"], "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"scoring": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"scoring": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"scoring": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"scoring": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def official_evaluation():
    if DEBUG or winner_run is None:
        raise RuntimeError("Complete the selected model's full refit before official evaluation.")
    recipe = plan["recipe_selected_on_tune"]
    if read_final_lock(recipe) != winner_run:
        raise RuntimeError("The refitted checkpoint differs from the selected lock.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    seen_names, unseen_names = map(set, filenames)
    if seen_names & unseen_names or set(train_full.image) & (seen_names | unseen_names):
        raise ValueError("Training and official split filenames overlap.")
    predictions = predict_recipe(recipe, winner_run, pairs, filenames)
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    report, predicted_seen = score_trial(winner_run, pairs, predictions, y_seen, y_unseen)
    summary = {"selection": winner_run, "recipe_selected_on_tune": recipe,
               "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    for key, values in report.items():
        summary[key] = [{"recipe": recipe, **row} for row in values]
    winners = {f"{recipe}_{rule}": values for rule, values in predicted_seen.items()}
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()

## 7. Save

Results are saved in `results/approach2_paired_disease_attention/<run_id>/{development,final,official}/`. The full model is saved in `checkpoints/approach2_paired_disease_attention/<run_id>/model.pt`.

Read `development/raw_check_deltas.csv` to isolate patch attention and paired supervision. Read `official/official_raw_results.csv` for the winning checkpoint's 333-way raw result. The historical Baseline 2 result should be labeled with its own training protocol. Only the winner is refit and tested officially.

In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    for mode, part in table.groupby("scoring", sort=False):
        part.to_csv(OFFICIAL_DIR / f"official_{mode}_results.csv", index=False)
    for name in ("restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['recipe']}_{row['scoring']}.json", row)
    atomic_json(OFFICIAL_DIR / "official_report.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    columns = ["recipe", "paired_supervision", "gamma", "epoch", "selected_on_tune",
               "seen_top1", "unseen_top1", "overall_top1", "seen_top5", "unseen_top5", "overall_top5", "harmonic_mean"]
    for mode, part in table.groupby("scoring", sort=False):
        print(f"{mode.capitalize()} results")
        display(part[columns])
    recipe = plan["recipe_selected_on_tune"]
    print(recipe, "checkpoint:", (CHECKPOINT_DIR / "model.pt").resolve())
    print("Development results:", DEVELOPMENT_DIR.resolve())
    print("Final selection and refit:", FINAL_DIR.resolve())
    print("Official results:", OFFICIAL_DIR.resolve())
else:
    print("Debug logs:", DEVELOPMENT_DIR.resolve())